In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# CONFIG
# ============================================================

BRONZE_TABLE = "lh_airops_bronze.brz_bts_flights"
SILVER_INPUT_TABLE = "slv_flights"

ACCEPTED_TABLE = "slv_flights_validated"
QUARANTINE_TABLE = "slv_flights_quarantine"
METRICS_TABLE = "slv_flights_dq_metrics"

BATCH_KEY = "bts_reporting_carrier_ontime|2026|04"
EXPECTED_INPUT_ROWS = 597_919

FLIGHT_KEY_VERSION = "flight_key_v1"
DQ_RULESET_VERSION = "dq_rules_v1"

KEY_FIELDS = [
    "flight_date",
    "reporting_airline",
    "flight_number",
    "origin",
    "dest",
    "crs_dep_time_hhmm",
]

LINEAGE_COLS = [
    "_bronze_run_id",
    "_bronze_load_id",
    "_bronze_batch_key",
    "_bronze_contract_version",
    "_bronze_source_name",
    "_bronze_source_file_name",
    "_bronze_source_hash",
    "_bronze_ingested_at_utc",
    "_bronze_load_year",
    "_bronze_load_month",
]

EMPTY_ARRAY = F.expr("cast(array() as array<string>)")


# ============================================================
# HELPERS
# ============================================================

def reason_if(condition, code):
    """
    Return [code] if a DQ rule fails, otherwise [].
    A row can therefore carry more than one reason.
    """
    return (
        F.when(condition, F.array(F.lit(code)))
         .otherwise(EMPTY_ARRAY)
    )


def flight_key_material():
    """
    Canonical scheduled-flight business identity.
    """
    return F.concat(
        F.lit("flight_date="),
        F.date_format("flight_date", "yyyy-MM-dd"),

        F.lit("|reporting_airline="),
        F.upper(F.trim(F.col("reporting_airline"))),

        F.lit("|flight_number="),
        F.col("flight_number").cast("string"),

        F.lit("|origin="),
        F.upper(F.trim(F.col("origin"))),

        F.lit("|dest="),
        F.upper(F.trim(F.col("dest"))),

        F.lit("|crs_dep_time_hhmm="),
        F.lpad(
            F.col("crs_dep_time_hhmm").cast("string"),
            4,
            "0"
        )
    )


def flight_key_expr():
    return F.sha2(
        flight_key_material(),
        256
    )


# ============================================================
# 1. READ BRONZE + TASK-5 SILVER
# ============================================================

bronze = (
    spark.table(BRONZE_TABLE)
         .filter(
             F.col("_bronze_batch_key") == BATCH_KEY
         )
)

silver = (
    spark.table(SILVER_INPUT_TABLE)
         .filter(
             F.col("_bronze_batch_key") == BATCH_KEY
         )
)

bronze_rows = bronze.count()
silver_rows = silver.count()

print(f"Bronze rows:       {bronze_rows:,}")
print(f"Silver input rows: {silver_rows:,}")

assert bronze_rows == EXPECTED_INPUT_ROWS, (
    f"STOP: Bronze expected {EXPECTED_INPUT_ROWS:,}, "
    f"found {bronze_rows:,}"
)

assert silver_rows == EXPECTED_INPUT_ROWS, (
    f"STOP: Silver expected {EXPECTED_INPUT_ROWS:,}, "
    f"found {silver_rows:,}"
)

assert bronze_rows == silver_rows, (
    "STOP: Bronze/Silver reconciliation was broken "
    "before Task 10 started."
)


# ============================================================
# 2. VERIFY REQUIRED COLUMNS
# ============================================================

required_cols = KEY_FIELDS + LINEAGE_COLS

missing_cols = [
    c for c in required_cols
    if c not in silver.columns
]

assert not missing_cols, (
    f"STOP: required columns missing: {missing_cols}"
)


# ============================================================
# 3. STRUCTURAL DATA-QUALITY RULES
# ============================================================

crs_time = F.col("crs_dep_time_hhmm")

valid_crs_time = (
    (crs_time == 2400)
    |
    (
        (crs_time >= 0)
        & (crs_time <= 2359)
        & ((crs_time % 100) < 60)
    )
)


# Check whether ANY required lineage field is NULL.

missing_lineage = None

for c in LINEAGE_COLS:

    condition = F.col(c).isNull()

    if missing_lineage is None:
        missing_lineage = condition
    else:
        missing_lineage = (
            missing_lineage | condition
        )


dq_reasons = F.concat(

    reason_if(
        F.col("flight_date").isNull(),
        "MISSING_FLIGHT_DATE"
    ),

    reason_if(
        F.col("reporting_airline").isNull()
        | (F.length(F.trim("reporting_airline")) == 0),
        "MISSING_REPORTING_AIRLINE"
    ),

    reason_if(
        F.col("flight_number").isNull(),
        "MISSING_FLIGHT_NUMBER"
    ),

    reason_if(
        F.col("flight_number").isNotNull()
        & (F.col("flight_number") <= 0),
        "INVALID_FLIGHT_NUMBER"
    ),

    reason_if(
        F.col("origin").isNull()
        | (F.length(F.trim("origin")) == 0),
        "MISSING_ORIGIN"
    ),

    reason_if(
        F.col("dest").isNull()
        | (F.length(F.trim("dest")) == 0),
        "MISSING_DEST"
    ),

    reason_if(
        crs_time.isNull(),
        "MISSING_CRS_DEP_TIME"
    ),

    reason_if(
        crs_time.isNotNull()
        & (~valid_crs_time),
        "INVALID_CRS_DEP_TIME"
    ),

    reason_if(
        F.col("flight_date").isNotNull()
        & (
            (F.year("flight_date") != 2026)
            | (F.month("flight_date") != 4)
        ),
        "FLIGHT_DATE_OUTSIDE_BATCH"
    ),

    reason_if(
        missing_lineage,
        "MISSING_REQUIRED_LINEAGE"
    ),
)


with_dq = (
    silver
    .withColumn(
        "_dq_reason_codes",
        dq_reasons
    )
    .withColumn(
        "_dq_ruleset_version",
        F.lit(DQ_RULESET_VERSION)
    )
)


base_valid = (
    with_dq
    .filter(
        F.size("_dq_reason_codes") == 0
    )
)

base_invalid = (
    with_dq
    .filter(
        F.size("_dq_reason_codes") > 0
    )
)


base_valid_rows = base_valid.count()
base_invalid_rows = base_invalid.count()

print(f"Base-valid rows:   {base_valid_rows:,}")
print(f"Base-invalid rows: {base_invalid_rows:,}")


# ============================================================
# 4. CREATE DETERMINISTIC flight_key
# ============================================================

keyed = (
    base_valid
    .withColumn(
        "flight_key",
        flight_key_expr()
    )
    .withColumn(
        "_flight_key_version",
        F.lit(FLIGHT_KEY_VERSION)
    )
)


null_keys = (
    keyed
    .filter(
        F.col("flight_key").isNull()
    )
    .count()
)

assert null_keys == 0, (
    f"STOP: {null_keys:,} valid rows generated NULL keys"
)


# ============================================================
# 5. PROVE HASH PRESERVES BUSINESS GRAIN
# ============================================================

distinct_business_keys = (
    keyed
    .select(*KEY_FIELDS)
    .distinct()
    .count()
)

distinct_hash_keys = (
    keyed
    .select("flight_key")
    .distinct()
    .count()
)

print(
    f"Distinct composite business keys: "
    f"{distinct_business_keys:,}"
)

print(
    f"Distinct SHA-256 flight_key:       "
    f"{distinct_hash_keys:,}"
)


assert distinct_business_keys == distinct_hash_keys, (
    "STOP: hashed flight_key count does not match "
    "the composite business-key grain."
)


# ============================================================
# 6. CREATE STABLE ROW FINGERPRINT
# ============================================================
#
# Used only as a deterministic tie-breaker if two rows have
# the same business key.
#
# Exclude Silver transformation timestamp because it changes
# each time Task 5 is rerun.

fingerprint_cols = [
    c for c in silver.columns
    if c != "_silver_transformed_at_utc"
]


keyed = keyed.withColumn(
    "_row_fingerprint",

    F.sha2(
        F.to_json(
            F.struct(
                *[
                    F.col(c)
                    for c in fingerprint_cols
                ]
            ),
            options={
                "ignoreNullFields": "false"
            }
        ),
        256
    )
)


# ============================================================
# 7. DETERMINISTIC DUPLICATE DETECTION
# ============================================================

group_window = (
    Window
    .partitionBy("flight_key")
)

dedupe_window = (
    Window
    .partitionBy("flight_key")
    .orderBy(

        F.col(
            "_bronze_ingested_at_utc"
        ).asc_nulls_last(),

        F.col(
            "_bronze_load_id"
        ).asc_nulls_last(),

        F.col(
            "_bronze_run_id"
        ).asc_nulls_last(),

        F.col(
            "_row_fingerprint"
        ).asc()
    )
)


ranked = (
    keyed

    .withColumn(
        "_duplicate_group_size",
        F.count(
            F.lit(1)
        ).over(group_window)
    )

    .withColumn(
        "_duplicate_rank",
        F.row_number().over(
            dedupe_window
        )
    )
)


# ============================================================
# 8. ACCEPT EXACTLY ONE SURVIVOR PER flight_key
# ============================================================

accepted = (
    ranked

    .filter(
        F.col("_duplicate_rank") == 1
    )

    .withColumn(
        "_dq_status",
        F.lit("ACCEPTED")
    )

    .withColumn(
        "_dq_checked_at_utc",
        F.current_timestamp()
    )
)


# ============================================================
# 9. QUARANTINE DUPLICATE EXTRAS
# ============================================================

duplicate_quarantine = (
    ranked

    .filter(
        F.col("_duplicate_rank") > 1
    )

    .withColumn(
        "_dq_reason_codes",

        F.concat(
            F.col("_dq_reason_codes"),
            F.array(
                F.lit(
                    "DUPLICATE_FLIGHT_KEY"
                )
            )
        )
    )

    .withColumn(
        "_dq_status",
        F.lit("QUARANTINED")
    )

    .withColumn(
        "_dq_checked_at_utc",
        F.current_timestamp()
    )
)


duplicate_rows = (
    duplicate_quarantine.count()
)

print(
    f"Duplicate extras quarantined: "
    f"{duplicate_rows:,}"
)


# ============================================================
# 10. PREPARE STRUCTURAL-INVALID QUARANTINE
# ============================================================

invalid_quarantine = (
    base_invalid

    .withColumn(
        "flight_key",
        F.lit(None).cast("string")
    )

    .withColumn(
        "_flight_key_version",
        F.lit(FLIGHT_KEY_VERSION)
    )

    .withColumn(
        "_row_fingerprint",
        F.lit(None).cast("string")
    )

    .withColumn(
        "_duplicate_group_size",
        F.lit(None).cast("long")
    )

    .withColumn(
        "_duplicate_rank",
        F.lit(None).cast("int")
    )

    .withColumn(
        "_dq_status",
        F.lit("QUARANTINED")
    )

    .withColumn(
        "_dq_checked_at_utc",
        F.current_timestamp()
    )
)


quarantine = (
    invalid_quarantine

    .unionByName(
        duplicate_quarantine,
        allowMissingColumns=True
    )
)


# ============================================================
# 11. ACCEPTED-GRAIN VALIDATION
# ============================================================

accepted_rows = accepted.count()
quarantine_rows = quarantine.count()


accepted_distinct_keys = (
    accepted
    .select("flight_key")
    .distinct()
    .count()
)


accepted_null_keys = (
    accepted
    .filter(
        F.col("flight_key").isNull()
    )
    .count()
)


accepted_duplicate_keys = (
    accepted

    .groupBy("flight_key")

    .count()

    .filter(
        F.col("count") > 1
    )

    .count()
)


assert accepted_null_keys == 0

assert accepted_duplicate_keys == 0

assert (
    accepted_rows
    ==
    accepted_distinct_keys
), (
    "STOP: accepted Silver grain is not unique."
)


# ============================================================
# 12. BRONZE → ACCEPTED + QUARANTINE RECONCILIATION
# ============================================================

assert (
    silver_rows
    ==
    accepted_rows + quarantine_rows
), (
    f"STOP: Silver reconciliation failed: "
    f"{silver_rows:,} != "
    f"{accepted_rows:,} + {quarantine_rows:,}"
)


assert (
    bronze_rows
    ==
    accepted_rows + quarantine_rows
), (
    f"STOP: Bronze reconciliation failed: "
    f"{bronze_rows:,} != "
    f"{accepted_rows:,} + {quarantine_rows:,}"
)


# ============================================================
# 13. RECOMPUTE KEYS TO PROVE DETERMINISM
# ============================================================

key_mismatch_rows = (
    accepted

    .withColumn(
        "_recomputed_flight_key",
        flight_key_expr()
    )

    .filter(
        F.col("flight_key")
        !=
        F.col("_recomputed_flight_key")
    )

    .count()
)


assert key_mismatch_rows == 0, (
    f"STOP: deterministic key mismatch count = "
    f"{key_mismatch_rows:,}"
)


# ============================================================
# 14. SHOW QUARANTINE REASONS
# ============================================================

reason_summary = (
    quarantine

    .select(
        F.explode(
            "_dq_reason_codes"
        ).alias("reason_code")
    )

    .groupBy(
        "reason_code"
    )

    .count()

    .orderBy(
        F.desc("count"),
        F.asc("reason_code")
    )
)


print("\nQuarantine reason summary:")

reason_summary.show(
    truncate=False
)


# ============================================================
# 15. WRITE ACCEPTED SILVER
# ============================================================

(
    accepted.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        ACCEPTED_TABLE
    )
)


# ============================================================
# 16. WRITE QUARANTINE
# ============================================================

(
    quarantine.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        QUARANTINE_TABLE
    )
)


# ============================================================
# 17. POST-WRITE VALIDATION
# ============================================================

accepted_post = (
    spark.table(
        ACCEPTED_TABLE
    )
)

quarantine_post = (
    spark.table(
        QUARANTINE_TABLE
    )
)


accepted_post_rows = (
    accepted_post.count()
)

quarantine_post_rows = (
    quarantine_post.count()
)


assert (
    accepted_post_rows
    ==
    accepted_rows
)

assert (
    quarantine_post_rows
    ==
    quarantine_rows
)

assert (
    bronze_rows
    ==
    accepted_post_rows
    +
    quarantine_post_rows
)


# ============================================================
# 18. WRITE DQ METRICS
# ============================================================

metrics = (
    spark.range(1)

    .select(

        F.current_timestamp()
        .alias(
            "measured_at_utc"
        ),

        F.lit(BATCH_KEY)
        .alias(
            "batch_key"
        ),

        F.lit(
            FLIGHT_KEY_VERSION
        )
        .alias(
            "flight_key_version"
        ),

        F.lit(
            DQ_RULESET_VERSION
        )
        .alias(
            "dq_ruleset_version"
        ),

        F.lit(
            bronze_rows
        )
        .cast("long")
        .alias(
            "bronze_rows"
        ),

        F.lit(
            silver_rows
        )
        .cast("long")
        .alias(
            "silver_input_rows"
        ),

        F.lit(
            base_invalid_rows
        )
        .cast("long")
        .alias(
            "structural_invalid_rows"
        ),

        F.lit(
            duplicate_rows
        )
        .cast("long")
        .alias(
            "duplicate_rows_quarantined"
        ),

        F.lit(
            accepted_rows
        )
        .cast("long")
        .alias(
            "accepted_rows"
        ),

        F.lit(
            quarantine_rows
        )
        .cast("long")
        .alias(
            "quarantine_rows"
        ),

        F.lit(
            accepted_distinct_keys
        )
        .cast("long")
        .alias(
            "accepted_distinct_flight_keys"
        ),

        F.lit(
            key_mismatch_rows
        )
        .cast("long")
        .alias(
            "key_recompute_mismatches"
        ),

        F.lit(
            bronze_rows
            ==
            accepted_rows
            +
            quarantine_rows
        )
        .alias(
            "reconciliation_pass"
        ),

        F.lit(
            accepted_rows
            ==
            accepted_distinct_keys
        )
        .alias(
            "accepted_key_uniqueness_pass"
        )
    )
)


(
    metrics.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        METRICS_TABLE
    )
)


# ============================================================
# 19. FINAL EVIDENCE
# ============================================================

print("\n" + "=" * 78)

print(
    "AIROPS 360 - TASK 10 "
    "FLIGHT KEY + DEDUPE + DQ GATE"
)

print("=" * 78)

print(
    f"Bronze control rows:          "
    f"{bronze_rows:,}"
)

print(
    f"Silver input rows:            "
    f"{silver_rows:,}"
)

print(
    f"Structural-invalid rows:      "
    f"{base_invalid_rows:,}"
)

print(
    f"Duplicate rows quarantined:   "
    f"{duplicate_rows:,}"
)

print(
    f"Accepted rows:                "
    f"{accepted_rows:,}"
)

print(
    f"Quarantine rows:              "
    f"{quarantine_rows:,}"
)

print(
    f"Accepted distinct flight_key: "
    f"{accepted_distinct_keys:,}"
)

print(
    f"Accepted NULL flight_key:     "
    f"{accepted_null_keys:,}"
)

print(
    f"Accepted duplicate keys:      "
    f"{accepted_duplicate_keys:,}"
)

print(
    f"Key recompute mismatches:      "
    f"{key_mismatch_rows:,}"
)

print(
    f"Reconciliation:               "
    f"{bronze_rows:,} = "
    f"{accepted_rows:,} + "
    f"{quarantine_rows:,}"
)

print(
    f"Accepted table:               "
    f"{ACCEPTED_TABLE}"
)

print(
    f"Quarantine table:             "
    f"{QUARANTINE_TABLE}"
)

print(
    f"Metrics table:                "
    f"{METRICS_TABLE}"
)

print("=" * 78)

print("\nTASK 10 STATUS: PASS")


# ============================================================
# 20. HUMAN-READABLE OUTPUT
# ============================================================

print("\nAccepted sample:")

display(
    accepted_post.select(
        "flight_key",
        "flight_date",
        "reporting_airline",
        "flight_number",
        "origin",
        "dest",
        "crs_dep_time_hhmm",
        "_duplicate_group_size",
        "_dq_status",
        "_flight_key_version",
        "_dq_ruleset_version",
    ).limit(20)
)


print("\nQuarantine sample:")

display(
    quarantine_post.select(
        "flight_key",
        "flight_date",
        "reporting_airline",
        "flight_number",
        "origin",
        "dest",
        "crs_dep_time_hhmm",
        "_duplicate_group_size",
        "_duplicate_rank",
        "_dq_status",
        "_dq_reason_codes",
    ).limit(20)
)


print("\nDQ metrics:")

display(
    spark.table(
        METRICS_TABLE
    )
)

StatementMeta(, 52e32b01-9d9b-4edf-befe-008f6c19ff11, 3, Finished, Available, Finished, True)

Bronze rows:       597,919
Silver input rows: 597,919
Base-valid rows:   597,919
Base-invalid rows: 0
Distinct composite business keys: 597,919
Distinct SHA-256 flight_key:       597,919
Duplicate extras quarantined: 0

Quarantine reason summary:
+-----------+-----+
|reason_code|count|
+-----------+-----+
+-----------+-----+


AIROPS 360 - TASK 10 FLIGHT KEY + DEDUPE + DQ GATE
Bronze control rows:          597,919
Silver input rows:            597,919
Structural-invalid rows:      0
Duplicate rows quarantined:   0
Accepted rows:                597,919
Quarantine rows:              0
Accepted distinct flight_key: 597,919
Accepted NULL flight_key:     0
Accepted duplicate keys:      0
Key recompute mismatches:      0
Reconciliation:               597,919 = 597,919 + 0
Accepted table:               slv_flights_validated
Quarantine table:             slv_flights_quarantine
Metrics table:                slv_flights_dq_metrics

TASK 10 STATUS: PASS

Accepted sample:


SynapseWidget(Synapse.DataFrame, 491ddb2f-1678-4184-aa58-8c030ac9707a)


Quarantine sample:


SynapseWidget(Synapse.DataFrame, 8ade5114-e7ab-4591-90a1-7c45128eeee8)


DQ metrics:


SynapseWidget(Synapse.DataFrame, f6dda9b3-37a4-42c8-bb80-5b6c000bfc8f)